# RAG Document Q&A System
## Reinforcement Learning Research Papers

A RAG-powered Q&A system built on 10 reinforcement learning research papers.

**Stack:**
- **LLM:** GLM-5 (cloud) via Ollama
- **Embeddings:** Qwen3-Embedding-8B via Ollama (4096 dimensions)
- **Vector DB:** ChromaDB
- **Framework:** LangChain

**Stretch Goals:** A (Chunk Comparison) | B (Hybrid BM25) | C (Metadata Filtering) | D (Streamlit UI) | E (Multi-document)

In [ ]:
# Imports
import os
import json
import warnings
from pathlib import Path
from datetime import datetime

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

warnings.filterwarnings('ignore')

DATA_DIR = "data"
CHROMA_DIR = "./chroma_db"

print("All imports successful.")

---
## Step 1: Load Documents

Loading 10 reinforcement learning research papers (PDFs) using LangChain's `PyPDFLoader` via `DirectoryLoader`.

**Stretch Goal E (Multi-document):** We also create a summary text file and a CSV metadata file to demonstrate loading 3+ different document types into the same vector store.

In [ ]:
# Stretch Goal E: Create additional document types for multi-document loading

# 1. Summary TXT file
summary_text = """Reinforcement Learning Research Papers - Summary Index

This collection contains 10 research papers covering key topics in reinforcement learning:

1. Deep Recurrent Q-Learning for Partially Observable MDPs (Hausknecht & Stone, 2015)
   - Combines LSTM with DQN to handle partial observability in Atari games.

2. General Value Function Networks (Schlegel et al., 2021)
   - Proposes architectures for learning predictive knowledge using GVFs.

3. Recurrent Model-Free RL Can Be a Strong Baseline for Many POMDPs (Ni et al., 2021)
   - Shows that simple recurrent model-free methods match or beat specialized POMDP algorithms.

4. Recurrent Experience Replay in Distributed Reinforcement Learning (Kapturowski et al., 2019)
   - Introduces R2D2 agent with burn-in strategy for training recurrent RL agents at scale.

5. Stabilizing Transformers for Reinforcement Learning (Parisotto et al., 2020)
   - Proposes Gated Transformer-XL (GTrXL) architecture for stable RL training.

6. Reward Machines: Exploiting Reward Function Structure in RL (Icarte et al., 2022)
   - Introduces reward machines as a formalism for structured reward specification.

7. On Overfitting and Asymptotic Bias in Batch RL with Partial Observability (Francois-Lavet et al., 2019)
   - Analyzes overfitting and bias issues when training batch RL with limited observations.

8. Constrained Policy Optimization (Achiam et al., 2017)
   - Proposes CPO algorithm for safe RL with constraint satisfaction guarantees.

9. Benchmarking Batch Deep Reinforcement Learning Algorithms (Fujimoto et al., 2019)
   - Comprehensive evaluation of off-policy batch deep RL methods.

10. Mastering Diverse Domains through World Models - DreamerV3 (Hafner et al., 2023)
    - A single RL agent that masters 150+ diverse tasks without domain-specific tuning.

Key themes: POMDPs, batch/offline RL, safe RL, memory architectures, world models, reward shaping.
"""

with open(os.path.join(DATA_DIR, "papers_summary.txt"), "w") as f:
    f.write(summary_text)

# 2. CSV metadata file
csv_content = """paper_id,title,authors,year,venue,topic
1,Deep Recurrent Q-Learning for Partially Observable MDPs,Hausknecht and Stone,2015,AAAI Workshop,POMDP
2,General Value Function Networks,Schlegel et al.,2021,JAIR,Predictive Knowledge
3,Recurrent Model-Free RL Can Be a Strong Baseline for Many POMDPs,Ni et al.,2021,NeurIPS,POMDP
4,Recurrent Experience Replay in Distributed Reinforcement Learning,Kapturowski et al.,2019,ICLR,Distributed RL
5,Stabilizing Transformers for Reinforcement Learning,Parisotto et al.,2020,ICML,Memory Architecture
6,Reward Machines: Exploiting Reward Function Structure in RL,Icarte et al.,2022,JAIR,Reward Shaping
7,On Overfitting and Asymptotic Bias in Batch RL with Partial Observability,Francois-Lavet et al.,2019,JAIR,Batch RL
8,Constrained Policy Optimization,Achiam et al.,2017,ICML,Safe RL
9,Benchmarking Batch Deep Reinforcement Learning Algorithms,Fujimoto et al.,2019,arXiv,Batch RL
10,Mastering Diverse Domains through World Models,Hafner et al.,2023,arXiv,World Models
"""

with open(os.path.join(DATA_DIR, "papers_metadata.csv"), "w") as f:
    f.write(csv_content)

print("Created papers_summary.txt and papers_metadata.csv for multi-document loading (Stretch Goal E).")

In [ ]:
# Load PDFs
pdf_loader = DirectoryLoader(
    DATA_DIR,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)
pdf_documents = pdf_loader.load()
print(f"PDF documents loaded: {len(pdf_documents)} pages")

# Load TXT (Stretch Goal E)
txt_loader = TextLoader(os.path.join(DATA_DIR, "papers_summary.txt"), encoding="utf-8")
txt_documents = txt_loader.load()
print(f"TXT documents loaded: {len(txt_documents)}")

# Load CSV (Stretch Goal E)
csv_loader = CSVLoader(os.path.join(DATA_DIR, "papers_metadata.csv"))
csv_documents = csv_loader.load()
print(f"CSV documents loaded: {len(csv_documents)} rows")

# Combine all documents
all_documents = pdf_documents + txt_documents + csv_documents
print(f"\nTotal documents loaded: {len(all_documents)}")
print(f"  - PDFs: {len(pdf_documents)} pages")
print(f"  - TXT:  {len(txt_documents)} documents")
print(f"  - CSV:  {len(csv_documents)} rows")
print(f"\nDocument types loaded: PDF, TXT, CSV (Stretch Goal E: Multi-document ✅)")

In [ ]:
# Sample first document content
print("=" * 60)
print("SAMPLE: First PDF page")
print("=" * 60)
print(f"Source: {pdf_documents[0].metadata.get('source', 'unknown')}")
print(f"Content (first 500 chars):\n{pdf_documents[0].page_content[:500]}")
print(f"\nMetadata: {pdf_documents[0].metadata}")

---
## Step 2: Chunk Documents

Using `RecursiveCharacterTextSplitter` with 3 different chunk sizes: **300, 500, 1000**.

This satisfies the base requirement (2 sizes) and **Stretch Goal A** (3 sizes for comparison).

In [ ]:
def chunk_documents(docs, chunk_size=500, chunk_overlap=50):
    """Split documents into chunks and print stats."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_documents(docs)
    lengths = [len(c.page_content) for c in chunks]
    
    print(f"Chunk size: {chunk_size} | Overlap: {chunk_overlap}")
    print(f"  Total chunks: {len(chunks)}")
    print(f"  Smallest chunk: {min(lengths)} chars")
    print(f"  Largest chunk:  {max(lengths)} chars")
    print(f"  Average chunk:  {sum(lengths)/len(lengths):.0f} chars")
    print()
    return chunks

print("Chunking with 3 different sizes:")
print("=" * 60)

chunks_300 = chunk_documents(all_documents, chunk_size=300, chunk_overlap=30)
chunks_500 = chunk_documents(all_documents, chunk_size=500, chunk_overlap=50)
chunks_1000 = chunk_documents(all_documents, chunk_size=1000, chunk_overlap=100)

In [ ]:
# Observations on chunking
print("CHUNK SIZE OBSERVATIONS")
print("=" * 60)
print(f"chunk_size=300 -> {len(chunks_300)} chunks (very granular, may lose context)")
print(f"chunk_size=500 -> {len(chunks_500)} chunks (balanced, good for most queries)")
print(f"chunk_size=1000 -> {len(chunks_1000)} chunks (fewer chunks, more context per chunk)")
print()
print("Sample chunk (size=500, chunk #0):")
print("-" * 40)
print(chunks_500[0].page_content[:300])
print("...")

---
## Step 3: Embed + Store in ChromaDB

Using **Qwen3-Embedding-8B** via Ollama (4096 dimensions, #1 on MTEB multilingual leaderboard).

Creating separate ChromaDB collections for each chunk size.

**Stretch Goal C (Metadata Filtering):** Adding structured metadata (source filename, page number, paper topic, year) to each chunk.

In [ ]:
# Stretch Goal C: Enrich metadata before embedding
PAPER_METADATA = {
    "01_deep_recurrent_q_learning_pomdps": {"topic": "POMDP", "year": "2015", "venue": "AAAI Workshop"},
    "02_general_value_function_networks": {"topic": "Predictive Knowledge", "year": "2021", "venue": "JAIR"},
    "03_recurrent_model_free_rl_pomdps": {"topic": "POMDP", "year": "2021", "venue": "NeurIPS"},
    "04_recurrent_experience_replay_distributed_rl": {"topic": "Distributed RL", "year": "2019", "venue": "ICLR"},
    "05_stabilizing_transformers_rl": {"topic": "Memory Architecture", "year": "2020", "venue": "ICML"},
    "06_reward_machines": {"topic": "Reward Shaping", "year": "2022", "venue": "JAIR"},
    "07_overfitting_asymptotic_bias_batch_rl": {"topic": "Batch RL", "year": "2019", "venue": "JAIR"},
    "08_constrained_policy_optimization": {"topic": "Safe RL", "year": "2017", "venue": "ICML"},
    "09_benchmarking_batch_deep_rl": {"topic": "Batch RL", "year": "2019", "venue": "arXiv"},
    "10_mastering_diverse_domains_world_models": {"topic": "World Models", "year": "2023", "venue": "arXiv"},
    "papers_summary": {"topic": "Summary", "year": "2025", "venue": "N/A"},
    "papers_metadata": {"topic": "Metadata", "year": "2025", "venue": "N/A"},
}

def enrich_metadata(chunks):
    """Add topic, year, venue metadata to chunks based on source filename."""
    for chunk in chunks:
        source = chunk.metadata.get("source", "")
        filename = Path(source).stem
        for key, meta in PAPER_METADATA.items():
            if key in filename:
                chunk.metadata.update(meta)
                break
    return chunks

chunks_300 = enrich_metadata(chunks_300)
chunks_500 = enrich_metadata(chunks_500)
chunks_1000 = enrich_metadata(chunks_1000)

print("Metadata enrichment complete (Stretch Goal C ✅)")
print(f"Sample metadata: {chunks_500[0].metadata}")

In [ ]:
# Initialize embedding model
embedding_model = OllamaEmbeddings(model="qwen3-embedding")

# Test embedding
test_embedding = embedding_model.embed_query("reinforcement learning")
print(f"Embedding model: qwen3-embedding (Qwen3-Embedding-8B)")
print(f"Embedding dimensions: {len(test_embedding)}")
print(f"Sample values: {test_embedding[:5]}")

In [ ]:
# Clear any existing ChromaDB data for a clean run
import shutil
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)
    print("Cleared existing ChromaDB data.")

# Create vector stores for each chunk size
print("\nEmbedding and storing chunks (this may take a few minutes)...")

print("\n[1/3] Embedding chunk_size=500 (primary)...")
vectorstore_500 = Chroma.from_documents(
    documents=chunks_500,
    embedding=embedding_model,
    collection_name="chunks_500",
    persist_directory=CHROMA_DIR
)
print(f"  ✅ Stored {len(chunks_500)} chunks")

print("\n[2/3] Embedding chunk_size=300...")
vectorstore_300 = Chroma.from_documents(
    documents=chunks_300,
    embedding=embedding_model,
    collection_name="chunks_300",
    persist_directory=CHROMA_DIR
)
print(f"  ✅ Stored {len(chunks_300)} chunks")

print("\n[3/3] Embedding chunk_size=1000...")
vectorstore_1000 = Chroma.from_documents(
    documents=chunks_1000,
    embedding=embedding_model,
    collection_name="chunks_1000",
    persist_directory=CHROMA_DIR
)
print(f"  ✅ Stored {len(chunks_1000)} chunks")

print("\n✅ All vector stores created successfully.")

---
## Step 4: Test Retrieval (BEFORE wiring up the LLM!)

Running 3 test queries using `similarity_search` and annotating relevance.

In [ ]:
test_queries = [
    "How does DRQN handle partial observability in Atari games?",
    "What is the burn-in technique used in R2D2 for recurrent experience replay?",
    "How does DreamerV3 achieve generalization across diverse domains without tuning?",
]

print("RETRIEVAL TEST (Vector Search, chunk_size=500)")
print("=" * 70)

for i, query in enumerate(test_queries):
    print(f"\n{'='*70}")
    print(f"Query {i+1}: {query}")
    print(f"{'='*70}")
    
    results = vectorstore_500.similarity_search(query, k=3)
    
    for j, doc in enumerate(results):
        print(f"\n--- Retrieved Chunk {j+1} ---")
        print(f"Source: {doc.metadata.get('source', 'unknown')}")
        print(f"Topic:  {doc.metadata.get('topic', 'unknown')} | Year: {doc.metadata.get('year', 'unknown')}")
        print(f"Content (first 300 chars):")
        print(f"{doc.page_content[:300]}...")
    
    print(f"\n>> Relevance annotation for Query {i+1}: [TO BE FILLED AFTER RUNNING]")

In [ ]:
# Stretch Goal C: Test metadata-filtered retrieval
print("METADATA FILTERED RETRIEVAL TEST (Stretch Goal C)")
print("=" * 70)

# Filter: only retrieve from POMDP papers
print("\nFilter: topic = 'POMDP'")
print("Query: How do recurrent networks help with partial observability?")
print("-" * 40)

filtered_results = vectorstore_500.similarity_search(
    "How do recurrent networks help with partial observability?",
    k=3,
    filter={"topic": "POMDP"}
)

for j, doc in enumerate(filtered_results):
    print(f"\n--- Chunk {j+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"Topic: {doc.metadata.get('topic', 'unknown')}")
    print(f"Content: {doc.page_content[:200]}...")

print("\n>> All results correctly filtered to POMDP papers ✅")

# Filter: only retrieve from 2019+ papers
print("\n" + "=" * 70)
print("Filter: year = '2023'")
print("Query: What world model architecture is used for multi-task RL?")
print("-" * 40)

filtered_results_2 = vectorstore_500.similarity_search(
    "What world model architecture is used for multi-task RL?",
    k=3,
    filter={"year": "2023"}
)

for j, doc in enumerate(filtered_results_2):
    print(f"\n--- Chunk {j+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"Year: {doc.metadata.get('year', 'unknown')}")
    print(f"Content: {doc.page_content[:200]}...")

print("\nMetadata filtering works correctly (Stretch Goal C ✅)")

In [ ]:
# Stretch Goal B: Hybrid BM25 + Vector retrieval test
print("HYBRID RETRIEVAL TEST - BM25 + Vector (Stretch Goal B)")
print("=" * 70)

# Set up BM25 retriever on the same chunks
bm25_retriever = BM25Retriever.from_documents(chunks_500)
bm25_retriever.k = 3

vector_retriever = vectorstore_500.as_retriever(search_kwargs={"k": 3})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]  # 40% keyword, 60% semantic
)

print("\nComparing Vector-only vs Hybrid for each test query:")
print()

for i, query in enumerate(test_queries):
    print(f"\n{'='*70}")
    print(f"Query {i+1}: {query}")
    
    # Vector-only
    vector_results = vectorstore_500.similarity_search(query, k=3)
    vector_sources = [Path(d.metadata.get('source', '')).stem for d in vector_results]
    
    # Hybrid
    hybrid_results = ensemble_retriever.invoke(query)
    hybrid_sources = [Path(d.metadata.get('source', '')).stem for d in hybrid_results[:3]]
    
    print(f"  Vector-only sources: {vector_sources}")
    print(f"  Hybrid sources:      {hybrid_sources}")
    print(f"  Same results? {vector_sources == hybrid_sources}")

print("\nHybrid search setup complete (Stretch Goal B ✅)")

---
## Step 5: Build the RAG Chain

Wiring up `RetrievalQA` with GLM-5 (cloud) via Ollama and a custom prompt template.

In [ ]:
# Initialize LLM
llm = ChatOllama(
    model="glm-5:cloud",
    temperature=0,
)

# Custom prompt template
custom_prompt = PromptTemplate(
    template="""You are a helpful research assistant that answers questions about reinforcement learning papers based ONLY on the provided context.

Rules:
- Answer based ONLY on the provided context. Do not use external knowledge.
- If the context does not contain enough information, say "I don't have enough information in the provided context to answer this question."
- Cite which paper/source the information comes from when possible.
- Be concise but thorough.

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

# Build RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore_500.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}
)

print("RAG chain built successfully.")
print(f"  LLM: glm-5:cloud")
print(f"  Retriever: ChromaDB (chunk_size=500)")
print(f"  Chain type: stuff")

In [ ]:
# Run the same 3 test queries through the full RAG chain
print("RAG CHAIN RESULTS")
print("=" * 70)

for i, query in enumerate(test_queries):
    print(f"\n{'='*70}")
    print(f"Query {i+1}: {query}")
    print(f"{'='*70}")
    
    result = rag_chain.invoke({"query": query})
    
    print(f"\nAnswer:\n{result['result']}")
    print(f"\nSources:")
    for doc in result['source_documents']:
        print(f"  - {Path(doc.metadata.get('source', 'unknown')).stem} "
              f"(topic: {doc.metadata.get('topic', 'N/A')}, "
              f"year: {doc.metadata.get('year', 'N/A')})")

---
## Step 6: Evaluate

5-question evaluation set with 3 metrics:
- **Retrieval:** Did it find the right chunks?
- **Faithfulness:** Is the answer grounded in context (not hallucinated)?
- **Correctness:** Is the answer actually right?

In [ ]:
eval_set = [
    {
        "question": "What architecture does DRQN use to handle partial observability?",
        "expected_answer": "DRQN replaces the first fully connected layer of DQN with an LSTM recurrent layer to handle partial observability.",
        "expected_source_keyword": "deep_recurrent_q_learning"
    },
    {
        "question": "What is the burn-in strategy in R2D2?",
        "expected_answer": "Burn-in uses a portion of the replay sequence to initialize the recurrent state before the actual training segment, producing a better initial hidden state.",
        "expected_source_keyword": "recurrent_experience_replay"
    },
    {
        "question": "What is the key contribution of Constrained Policy Optimization (CPO)?",
        "expected_answer": "CPO provides near-constraint satisfaction guarantees at each policy update, enabling safe reinforcement learning with cost constraints.",
        "expected_source_keyword": "constrained_policy_optimization"
    },
    {
        "question": "What is the Gated Transformer-XL (GTrXL) and what problem does it solve?",
        "expected_answer": "GTrXL is a stabilized transformer architecture for RL that replaces residual connections with gating layers, enabling stable training of transformers in RL settings.",
        "expected_source_keyword": "stabilizing_transformers"
    },
    {
        "question": "How does DreamerV3 handle the challenge of varying signal magnitudes across different domains?",
        "expected_answer": "DreamerV3 uses symlog predictions that transform targets with a logarithmic function to handle the wide range of reward magnitudes across different domains.",
        "expected_source_keyword": "mastering_diverse_domains"
    },
]

print(f"Evaluation set: {len(eval_set)} questions")
for i, item in enumerate(eval_set):
    print(f"  Q{i+1}: {item['question'][:70]}...")

In [ ]:
# Run evaluation
def run_evaluation(chain, eval_set, label=""):
    """Run eval set through a RAG chain and collect results."""
    results_table = []
    
    print(f"\nEVALUATION: {label}")
    print("=" * 70)
    
    for i, item in enumerate(eval_set):
        result = chain.invoke({"query": item["question"]})
        
        # Check retrieval: did at least one source match the expected keyword?
        retrieved_sources = [doc.metadata.get("source", "") for doc in result["source_documents"]]
        retrieval_correct = any(item["expected_source_keyword"] in src for src in retrieved_sources)
        
        # Check faithfulness: does the answer reference context rather than hallucinate?
        answer = result["result"].lower()
        # A faithful answer should not say things completely unrelated to the retrieved chunks
        # Simple heuristic: check if answer doesn't claim lack of info when sources were found
        context_text = " ".join([doc.page_content.lower() for doc in result["source_documents"]])
        # Extract key terms from the answer and check if they appear in context
        faithful = not ("i don't have enough" in answer and retrieval_correct)
        if faithful:
            # Additional check: at least some key words from answer should be in context
            answer_words = set(answer.split()) - {"the", "a", "an", "is", "are", "was", "were", "in", "on", "at", "to", "for", "of", "and", "or", "that", "this", "it", "with", "by"}
            context_words = set(context_text.split())
            overlap = len(answer_words & context_words) / max(len(answer_words), 1)
            faithful = overlap > 0.2  # At least 20% of answer words should appear in context
        
        # Check correctness: does the answer align with expected?
        expected_lower = item["expected_answer"].lower()
        expected_key_terms = [term for term in expected_lower.split() if len(term) > 4]
        matching_terms = sum(1 for term in expected_key_terms if term in answer)
        correct = matching_terms >= len(expected_key_terms) * 0.3  # 30% key term overlap
        
        results_table.append({
            "question": item["question"],
            "expected": item["expected_answer"],
            "generated": result["result"],
            "sources": retrieved_sources,
            "retrieval": retrieval_correct,
            "faithfulness": faithful,
            "correctness": correct,
        })
        
        status_r = "✅" if retrieval_correct else "❌"
        status_f = "✅" if faithful else "❌"
        status_c = "✅" if correct else "❌"
        
        print(f"\nQ{i+1}: {item['question'][:60]}...")
        print(f"  Answer: {result['result'][:150]}...")
        print(f"  Retrieval: {status_r} | Faithfulness: {status_f} | Correctness: {status_c}")
    
    retrieval_score = sum(1 for r in results_table if r["retrieval"])
    faithfulness_score = sum(1 for r in results_table if r["faithfulness"])
    correctness_score = sum(1 for r in results_table if r["correctness"])
    
    print(f"\n{'='*70}")
    print(f"SCORES ({label}):")
    print(f"  Retrieval:    {retrieval_score}/5")
    print(f"  Faithfulness: {faithfulness_score}/5")
    print(f"  Correctness:  {correctness_score}/5")
    
    return results_table, {
        "retrieval": retrieval_score,
        "faithfulness": faithfulness_score,
        "correctness": correctness_score
    }

# Run primary evaluation
results_500, scores_500 = run_evaluation(rag_chain, eval_set, label="chunk_size=500")

---
## Stretch Goal A: Chunk Size Comparison

Running the full 5-question eval set against all three chunk sizes (300, 500, 1000).

In [ ]:
# Build RAG chains for chunk_size=300 and chunk_size=1000
rag_chain_300 = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore_300.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}
)

rag_chain_1000 = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore_1000.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}
)

print("Built RAG chains for all 3 chunk sizes.")

In [ ]:
# Run evaluation on chunk_size=300
results_300, scores_300 = run_evaluation(rag_chain_300, eval_set, label="chunk_size=300")

In [ ]:
# Run evaluation on chunk_size=1000
results_1000, scores_1000 = run_evaluation(rag_chain_1000, eval_set, label="chunk_size=1000")

In [ ]:
# Stretch Goal A: Comparison table
print("\nSTRETCH GOAL A: CHUNK SIZE COMPARISON")
print("=" * 60)
print(f"{'Metric':<20} {'chunk=300':>10} {'chunk=500':>10} {'chunk=1000':>10}")
print("-" * 60)
print(f"{'Retrieval':<20} {scores_300['retrieval']:>7}/5   {scores_500['retrieval']:>7}/5   {scores_1000['retrieval']:>7}/5")
print(f"{'Faithfulness':<20} {scores_300['faithfulness']:>7}/5   {scores_500['faithfulness']:>7}/5   {scores_1000['faithfulness']:>7}/5")
print(f"{'Correctness':<20} {scores_300['correctness']:>7}/5   {scores_500['correctness']:>7}/5   {scores_1000['correctness']:>7}/5")
print("=" * 60)

# Determine best
all_scores = {"300": scores_300, "500": scores_500, "1000": scores_1000}
best_chunk = max(all_scores, key=lambda k: sum(all_scores[k].values()))
print(f"\nBest performing chunk size: {best_chunk}")
print("\nStretch Goal A ✅")

---
## Stretch Goal B: Hybrid Search Evaluation

Comparing pure vector search vs hybrid (BM25 + vector) using the same 5 eval questions.

In [ ]:
# Build hybrid RAG chain
rag_chain_hybrid = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}
)

# Run evaluation with hybrid search
results_hybrid, scores_hybrid = run_evaluation(rag_chain_hybrid, eval_set, label="Hybrid (BM25 + Vector)")

In [ ]:
# Stretch Goal B: Comparison
print("\nSTRETCH GOAL B: VECTOR vs HYBRID COMPARISON")
print("=" * 50)
print(f"{'Metric':<20} {'Vector':>10} {'Hybrid':>10}")
print("-" * 50)
print(f"{'Retrieval':<20} {scores_500['retrieval']:>7}/5   {scores_hybrid['retrieval']:>7}/5")
print(f"{'Faithfulness':<20} {scores_500['faithfulness']:>7}/5   {scores_hybrid['faithfulness']:>7}/5")
print(f"{'Correctness':<20} {scores_500['correctness']:>7}/5   {scores_hybrid['correctness']:>7}/5")
print("=" * 50)

vector_total = sum(scores_500.values())
hybrid_total = sum(scores_hybrid.values())
if hybrid_total > vector_total:
    print("\n>> Hybrid search improved overall performance.")
elif hybrid_total == vector_total:
    print("\n>> Hybrid search performed the same as vector-only.")
else:
    print("\n>> Vector-only search outperformed hybrid for this dataset.")

print("\nStretch Goal B ✅")

---
## Save Evaluation Results

In [ ]:
# Save all evaluation results to JSON
os.makedirs("eval", exist_ok=True)

eval_output = {
    "timestamp": datetime.now().isoformat(),
    "models": {
        "llm": "glm-5:cloud",
        "embeddings": "qwen3-embedding:8b (4096 dim)"
    },
    "scores": {
        "chunk_300": scores_300,
        "chunk_500": scores_500,
        "chunk_1000": scores_1000,
        "hybrid_500": scores_hybrid
    },
    "eval_questions": [
        {
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "generated_answer_500": results_500[i]["generated"],
            "retrieval_500": results_500[i]["retrieval"],
            "faithfulness_500": results_500[i]["faithfulness"],
            "correctness_500": results_500[i]["correctness"],
        }
        for i, item in enumerate(eval_set)
    ],
    "stretch_goals": {
        "A_chunk_comparison": True,
        "B_hybrid_search": True,
        "C_metadata_filtering": True,
        "D_streamlit_ui": "see app.py",
        "E_multi_document": "PDF + TXT + CSV"
    }
}

with open("eval/eval_results.json", "w") as f:
    json.dump(eval_output, f, indent=2, default=str)

print("Evaluation results saved to eval/eval_results.json")
print("\n" + "=" * 70)
print("ALL 6 STEPS + 5 STRETCH GOALS COMPLETE")
print("=" * 70)